In [1]:
pip install transformers datasets scikit-learn seaborn

In [2]:
import torch
from transformers import RobertaModel, RobertaTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {device}")

Device : cuda


In [4]:
# Load RoBERTa-base tokenizer and model from HuggingFace
# The tokenizer converts raw text to token IDs
# The model weights are frozen - only LoRA parameters will be trained
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('roberta-base')

# Print model config to verify architecture
print(model.config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public m

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [5]:
# Load SST-2 dataset from the GLUE benchmark
# SST-2 is a binary sentiment classification task (positive/negative)
dataset_sst2 = load_dataset('glue', 'sst2')
print(dataset_sst2)
print(dataset_sst2['train'][0])

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [6]:
# Load QNLI dataset from the GLUE benchmark
# QNLI is a question-answer inference task: given a question and a sentence,
# determine whether the sentence contains the answer to the question
dataset_qnli = load_dataset('glue', 'qnli')
print(dataset_qnli)
print(dataset_qnli['train'][0])

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

qnli/test-00000-of-00001.parquet:   0%|          | 0.00/877k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})
{'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}


In [7]:
def tokenize_sst2(examples):
    """
    Tokenize SST-2 examples.
    SST-2 has a single text input (sentence).
    """
    return tokenizer(
        examples['sentence'],
        truncation=True,      # Truncate sequences longer than max_length
        max_length=128,       # Maximum sequence length (paper uses 128)
        padding='max_length'  # Pad shorter sequences to max_length
    )

def tokenize_qnli(examples):
    """
    Tokenize QNLI examples.
    QNLI has two text inputs (question + sentence) that are concatenated
    by the tokenizer with a separator token.
    """
    return tokenizer(
        examples['question'],
        examples['sentence'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

In [8]:
# Apply tokenization to both datasets using batched processing for efficiency
# This adds 'input_ids' and 'attention_mask' columns to each dataset
dataset_sst2 = dataset_sst2.map(tokenize_sst2, batched=True)
dataset_qnli = dataset_qnli.map(tokenize_qnli, batched=True)

# Verify that tokenization columns were added correctly
print(dataset_sst2['train'].column_names)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

['sentence', 'label', 'idx', 'input_ids', 'attention_mask']


In [9]:
# Remove unnecessary columns and keep only: input_ids, attention_mask, label
# Then convert datasets to PyTorch tensor format
dataset_sst2 = dataset_sst2.remove_columns(['idx', 'sentence'])
dataset_sst2 = dataset_sst2.with_format('torch')

dataset_qnli = dataset_qnli.remove_columns(['idx', 'question', 'sentence'])
dataset_qnli = dataset_qnli.with_format('torch')

# Verify that only the necessary columns remain
print(dataset_sst2['train'].column_names)
print(dataset_qnli['train'].column_names)

['label', 'input_ids', 'attention_mask']
['label', 'input_ids', 'attention_mask']


In [10]:
def partition_dataset(dataset, num_clients):
    """
    Randomly partition a dataset into equal parts for each client.
    Each client receives a non-overlapping subset of the data.

    Args:
        dataset: HuggingFace dataset to partition
        num_clients: number of clients to split the data among

    Returns:
        list of dataset subsets, one per client
    """
    n = len(dataset)

    # Randomly shuffle indices to ensure each client gets a random subset
    indices = np.random.permutation(n)

    # Size of each client's partition
    size = n // num_clients
    partitions = []

    for i in range(num_clients):
        # Compute start and end indices for client i
        start = i * size
        end = (i + 1) * size
        partitions.append(dataset.select(indices[start:end]))

    return partitions

In [11]:
# Partition SST-2 among clients 0 and 1, QNLI among clients 2 and 3
# This simulates the heterogeneous federated learning setting from the paper:
# clients 0-1 share the same task (SST-2), clients 2-3 share a different task (QNLI)
sst2_partitions = partition_dataset(dataset_sst2['train'], num_clients=2)
qnli_partitions = partition_dataset(dataset_qnli['train'], num_clients=2)

# Map each client ID to its local dataset
client_datasets = {
    0: sst2_partitions[0],  # SST-2
    1: sst2_partitions[1],  # SST-2
    2: qnli_partitions[0],  # QNLI
    3: qnli_partitions[1],  # QNLI
}

# Verify partition sizes
for cid, ds in client_datasets.items():
    print(f"Client {cid} : {len(ds)} examples")

Client 0 : 33674 examples
Client 1 : 33674 examples
Client 2 : 52371 examples
Client 3 : 52371 examples


In [12]:
# Create a DataLoader for each client
# batch_size=32 (paper uses 128, reduced here for Colab T4 memory constraints)
# shuffle=True to ensure random ordering of samples during training
client_loaders = {}
for cid, ds in client_datasets.items():
    client_loaders[cid] = DataLoader(ds, batch_size=32, shuffle=True)

# Verify number of batches per client
print(f"Number of batches per client:")
for cid, loader in client_loaders.items():
    print(f"  Client {cid} : {len(loader)} batches")

Number of batches per client:
  Client 0 : 1053 batches
  Client 1 : 1053 batches
  Client 2 : 1637 batches
  Client 3 : 1637 batches


In [13]:
import math

def initialize_lora_params(rank=4):
    """
    Initialize LoRA parameters (A and B matrices) for all query and value
    projections across all 12 layers of RoBERTa-base.

    Returns a dictionary: layer_name -> {'A': tensor, 'B': tensor}
    """
    lora_params = {}

    for layer_idx in range(12):  # 12 transformer layers
        for proj in ['query', 'value']:
            key = f'layer_{layer_idx}_{proj}'

            # A : shape (rank, 768), initialized with kaiming uniform
            A = torch.empty(rank, 768)
            torch.nn.init.kaiming_uniform_(A, a=math.sqrt(5))

            # B : shape (768, rank), initialized to zero so that
            # the LoRA delta B@A = 0 at the start of training,
            # leaving the pretrained model unchanged (paper Section 3.1)
            B = torch.zeros(768, rank)

            lora_params[key] = {
                'A': A.to(device),
                'B': B.to(device)
            }

    return lora_params

In [14]:
# Verify LoRA parameter initialization for one client
# Expected: 24 layers (12 transformer layers x 2 projections: query and value)
# A shape: (rank, 768) = (4, 768)
# B shape: (768, rank) = (768, 4)
lora_params_client0 = initialize_lora_params(rank=4)
print(f"Number of LoRA layers: {len(lora_params_client0)}")
print(f"Shape of A: {lora_params_client0['layer_0_query']['A'].shape}")
print(f"Shape of B: {lora_params_client0['layer_0_query']['B'].shape}")

Number of LoRA layers: 24
Shape of A: torch.Size([4, 768])
Shape of B: torch.Size([768, 4])
Number of LoRA layers: 24
Shape of A: torch.Size([4, 768])
Shape of B: torch.Size([768, 4])


In [15]:
# Initialize independent LoRA parameters for each client
# Each client starts with the same initialization but will diverge during local training
# reflecting their different task distributions (SST-2 vs QNLI)
client_lora_params = {
    cid: initialize_lora_params(rank=4) for cid in range(4)
}

print(f"Number of clients: {len(client_lora_params)}")

Number of clients: 4
Number of clients: 4


In [16]:
def lora_forward(model, input_ids, attention_mask, lora_params, rank=4, lora_alpha=1.0):
    """
    Forward pass through RoBERTa with LoRA deltas applied to
    query and value projections.

    Instead of modifying weight.data (which breaks the computation graph),
    we use forward hooks to add the LoRA delta to the layer output.
    """
    scaling = lora_alpha / rank
    hooks = []

    def make_hook(key):
        def hook(module, input, output):
            # input[0] shape: (batch, seq_len, 768)
            A = lora_params[key]['A']
            B = lora_params[key]['B']
            # Compute LoRA delta and add to output
            lora_delta = input[0] @ A.T @ B.T * scaling
            return output + lora_delta
        return hook

    # Register hooks on query and value projections
    for layer_idx in range(12):
        for proj in ['query', 'value']:
            key = f'layer_{layer_idx}_{proj}'
            layer = model.encoder.layer[layer_idx].attention.self
            hook = getattr(layer, proj).register_forward_hook(make_hook(key))
            hooks.append(hook)

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    # Remove all hooks after forward pass
    for hook in hooks:
        hook.remove()

    return outputs

In [17]:
# Test lora_forward with a small batch to verify the implementation
# Expected output shape: (batch_size, max_length, hidden_size) = (32, 128, 768)
model = model.to(device)
model.eval()

# Take one batch from client 0
batch = next(iter(client_loaders[0]))
input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)

# Forward pass with LoRA (no gradient computation needed for testing)
with torch.no_grad():
    outputs = lora_forward(model, input_ids, attention_mask, client_lora_params[0])

print(f"Output shape: {outputs.last_hidden_state.shape}")

Output shape: torch.Size([32, 128, 768])
Output shape: torch.Size([32, 128, 768])


In [18]:
import torch.nn as nn

class ClassificationHead(nn.Module):
    """
    Simple classification head on top of RoBERTa.
    Takes the [CLS] token representation and predicts the class.

    The [CLS] token (index 0) aggregates the full sequence representation
    and is standard practice for classification tasks with BERT-based models.
    """
    def __init__(self, hidden_size=768, num_classes=2):
        super().__init__()
        # Linear layer mapping from hidden_size (768) to number of classes (2)
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, roberta_output):
        # Extract [CLS] token (index 0) from all sequences in the batch
        # Shape: (batch_size, hidden_size) = (32, 768)
        cls_token = roberta_output.last_hidden_state[:, 0, :]
        # Shape: (batch_size, num_classes) = (32, 2)
        return self.linear(cls_token)

# Each client has its own classification head trained on its local data
client_heads = {
    cid: ClassificationHead().to(device) for cid in range(4)
}

print(f"Number of classification heads: {len(client_heads)}")

Number of classification heads: 4


In [19]:
def train_client(model, lora_params, head, loader, num_epochs=2, lr=3e-4):
    """
    Train a single client for num_epochs epochs.
    Only LoRA parameters (A and B matrices) and the classification head
    are updated — the base RoBERTa weights remain frozen.

    Args:
        model: frozen RoBERTa base model
        lora_params: dict of LoRA A and B matrices for this client
        head: classification head for this client
        loader: DataLoader with the client's local data
        num_epochs: number of local training epochs (paper uses 2)
        lr: learning rate for AdamW optimizer (paper uses 3e-4)

    Returns:
        updated lora_params, loss history per epoch
    """
    # Convert A and B tensors to nn.Parameters so they can be optimized
    trainable_params = []
    for key in lora_params:
        lora_params[key]['A'] = nn.Parameter(lora_params[key]['A'])
        lora_params[key]['B'] = nn.Parameter(lora_params[key]['B'])
        trainable_params += [lora_params[key]['A'], lora_params[key]['B']]

    # Also optimize the classification head parameters
    trainable_params += list(head.parameters())

    # AdamW optimizer as used in the paper
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    criterion = nn.CrossEntropyLoss()
    loss_history = []

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()

            # Forward pass: RoBERTa with LoRA deltas applied
            outputs = lora_forward(model, input_ids, attention_mask, lora_params)

            # Classification: extract [CLS] token and predict class
            logits = head(outputs)

            # Compute cross-entropy loss, backpropagate, update LoRA params
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        loss_history.append(avg_loss)
        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

    return lora_params, loss_history

In [ ]:
# Initialization phase (Algorithm 1, lines 3-6):
# Each client independently trains its local LoRA for E=2 epochs
# This captures initial task-specific adaptations in the LoRA parameters
# which will be used by the server for clustering (Section 4.1)
task_names = {0: 'SST-2', 1: 'SST-2', 2: 'QNLI', 3: 'QNLI'}

for cid in range(4):
    print(f"Training client {cid} ({task_names[cid]})...")
    client_lora_params[cid], _ = train_client(
        model=model,
        lora_params=client_lora_params[cid],
        head=client_heads[cid],
        loader=client_loaders[cid],
        num_epochs=2,  # E=2 as in the paper
        lr=3e-4
    )

Training client 0 (SST-2)...


In [ ]:
def extract_b_matrices(client_lora_params):
    """
    Extract LoRA B matrices for all clients and all layers.

    Returns a dictionary: client_id -> list of B matrices (one per layer)
    """
    b_matrices = {}
    for cid, lora_params in client_lora_params.items():
        b_matrices[cid] = []
        for layer_idx in range(12):
            for proj in ['query', 'value']:
                key = f'layer_{layer_idx}_{proj}'
                # Extract B matrix and detach from computation graph
                B = lora_params[key]['B'].detach().cpu()
                b_matrices[cid].append(B)
    return b_matrices

In [ ]:
b_matrices = extract_b_matrices(client_lora_params)
print(f"Number of clients: {len(b_matrices)}")
print(f"Number of B matrices per client: {len(b_matrices[0])}")
print(f"Shape of one B matrix: {b_matrices[0][0].shape}")

In [ ]:
def compute_cosine_similarity(b_matrices, client_i, client_j):
    """
    Compute cosine similarity between two clients across all layers.

    Args:
        b_matrices: dict of B matrices per client
        client_i: first client ID
        client_j: second client ID

    Returns:
        list of cosine similarity values, one per layer
    """
    similarities = []
    for layer_idx in range(24):  # 24 B matrices per client
        b_i = b_matrices[client_i][layer_idx].flatten()
        b_j = b_matrices[client_j][layer_idx].flatten()

        # Compute cosine similarity between the two flattened B matrices
        sim = F.cosine_similarity(b_i.unsqueeze(0), b_j.unsqueeze(0))
        similarities.append(sim.item())

    return similarities

In [ ]:
sim_0_1 = compute_cosine_similarity(b_matrices, 0, 1)
sim_0_2 = compute_cosine_similarity(b_matrices, 0, 2)
print(f"Avg similarity client 0 vs 1 (same task SST-2): {np.mean(sim_0_1):.4f}")
print(f"Avg similarity client 0 vs 2 (different task): {np.mean(sim_0_2):.4f}")

In [ ]:
# Check if B matrices are non-zero after training
for cid in range(4):
    b = client_lora_params[cid]['layer_0_query']['B']
    print(f"Client {cid} | B mean: {b.mean().item():.6f} | B std: {b.std().item():.6f}")

In [ ]:
# Check if LoRA parameters have gradients
for cid in range(4):
    A = client_lora_params[cid]['layer_0_query']['A']
    B = client_lora_params[cid]['layer_0_query']['B']
    print(f"Client {cid} | A requires_grad: {A.requires_grad} | B requires_grad: {B.requires_grad}")

In [ ]:
# Check if gradients are computed after a forward pass
model.train()
batch = next(iter(client_loaders[0]))
input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)
labels = batch['label'].to(device)

outputs = lora_forward(model, input_ids, attention_mask, client_lora_params[0])
logits = client_heads[0](outputs)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()

B = client_lora_params[0]['layer_0_query']['B']
print(f"B gradient: {B.grad}")